# **Collaborative Filtering Recommendation System (V2)**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## **Data Laoder**

In [2]:
books = pd.read_csv("data/BX-Books.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)
users = pd.read_csv("data/BX-Users.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)
ratings = pd.read_csv("data/BX-Book-Ratings.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)

## **Data Preprocessing**

### Book data manipulation

In [3]:
books.shape

(271360, 8)

In [4]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

In [5]:
books.head(3)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...


In [6]:
# Choose needed columns only
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-L']]

# renaming those into model friendly way
books.rename(columns = {
    "ISBN": "isbn",
    "Book-Title": "title",
    "Book-Author": "author",
    "Year-Of-Publication": "year",
    "Publisher": "publisher",
    "Image-URL-L": "img_url"
}, inplace=True)

books.head(3)

,isbn,title,author,year,publisher,img_url
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...


### Users data manipulation

In [7]:
users.shape

(278858, 3)

In [8]:
users.head(3)

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN


In [9]:
# Here we need all columns, let rename them

users.rename(columns = {
    "User-ID": "user_id",
    "Location": "location",
    "Age": "age"
}, inplace=True)

users.head(3)

,user_id,location,age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN


### ratings data manipulation

In [10]:
ratings.shape

(1149780, 3)

In [11]:
ratings.columns

Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='object')

In [12]:
ratings.head(3)

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0


In [13]:
# Here we need all columns, let rename them

ratings.rename(columns = {
    "User-ID": "user_id",
    "ISBN": "isbn",
    "Book-Rating": "ratings"
}, inplace=True)

ratings.head(3)

,user_id,isbn,ratings
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0


In [14]:
# user 11676 has rated 13602 times OR different different books

ratings['user_id'].value_counts()

user_id
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
119573        1
276706        1
276697        1
276679        1
276676        1
Name: count, Length: 105283, dtype: int64

In [15]:
ratings['user_id'].unique().shape # No of unique users

(105283,)

In [16]:
# Converts counts into True/False
# x is a boolean Series, Index = user_id    (mask)
x = ratings['user_id'].value_counts() > 200
x

user_id
11676      True
198711     True
153662     True
98391      True
35859      True
          ...  
119573    False
276706    False
276697    False
276679    False
276676    False
Name: count, Length: 105283, dtype: bool

In [17]:
x[x].shape # x[x] > filter only True values

(899,)

In [18]:
y = x[x].index # gives only the user IDs (names), not True/False
y # list of users who have more than 200 ratings

Index([ 11676, 198711, 153662,  98391,  35859, 212898, 278418,  76352, 110973,
       235105,
       ...
       116122,  44296,  28634,  59727,  73681, 274808, 188951,   9856, 155916,
       268622],
      dtype='int64', name='user_id', length=899)

In [19]:
# we fetch users who rated more than 200 times as a list 'y', then we crate a boolean mask using '.isin()', then apply that to entire table
# ratings['user_id'].isin(y) >>> return True / False for each row
# finally we have a table 'filtered_ratings' only keeps users who are in y

filtered_ratings = ratings[ratings['user_id'].isin(y)]
print(filtered_ratings.shape)
filtered_ratings.head()

(526356, 3)


,user_id,isbn,ratings
1456,277427,002542730X,10
1457,277427,0026217457,0
1458,277427,003008685X,8
1459,277427,0030615321,0
1460,277427,0060002050,0


In [20]:
# Keep only rows where isbn exists in BOTH tables when merging
rating_with_books = filtered_ratings.merge(books, on = "isbn")

In [21]:
# How many times each book got raitings

num_ratings = rating_with_books.groupby('title')['ratings'].count().reset_index()
num_ratings.sample(5) # How many people rated each book

,title,ratings
97754,Resurrecting Ravana (Buffy the Vampire Slayer),8
92539,"Poem a Day, Vol. 1",2
129061,The Last Mortal Generation: How Science Will A...,1
92740,Polaroids from the Dead,4
22477,Celebrate the Sun: A Love Story,1


In [22]:
num_ratings.rename(columns={'ratings' : 'num_of_ratings'}, inplace=True)
num_ratings.head()

,title,num_of_ratings
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [23]:
# for each row ( for each book) we have a num_of_ratings value, since rating_with_books had same in book servral rows each somtimes same 
# num_of_ratings value can be several palces bcs user_id + title both together unique eg :
# 277427 002542730X 10 Politically Correct.. James Finn.. ... 82
# 52614  002542730X 5  Politically Correct.. James Finn.. ... 82

final_ratings = rating_with_books.merge(num_ratings, on="title")
final_ratings.head(5) 

,user_id,isbn,ratings,title,author,year,publisher,img_url,num_of_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,82
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...,7
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,http://images.amazon.com/images/P/003008685X.0...,1
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,http://images.amazon.com/images/P/0030615321.0...,1
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,http://images.amazon.com/images/P/0060002050.0...,13


#### Consider only books that has rated more than 50 users

In [54]:
mask = final_ratings['num_of_ratings'] >= 50
final_ratings = final_ratings[mask]

In [28]:
print(final_ratings.shape)
final_ratings.sample(5)

(61853, 9)


,user_id,isbn,ratings,title,author,year,publisher,img_url,num_of_ratings
231466,131837,0425175405,0,Black Notice,Patricia Daniels Cornwell,2000,Berkley Publishing Group,http://images.amazon.com/images/P/0425175405.0...,123
107038,56856,034538184X,0,Degree of Guilt,Richard North Patterson,1995,Ballantine Books,http://images.amazon.com/images/P/034538184X.0...,60
316977,184299,0451184963,8,Insomnia,Stephen King,1995,Signet Book,http://images.amazon.com/images/P/0451184963.0...,105
182492,104636,0449702545,9,Homecoming,Cynthia Voigt,1990,Fawcett Books,http://images.amazon.com/images/P/0449702545.0...,61
384051,224430,0385508417,0,Skipping Christmas,JOHN GRISHAM,2002,Doubleday,http://images.amazon.com/images/P/0385508417.0...,103


In [29]:
final_ratings.drop_duplicates(['user_id', 'title'], inplace=True)
final_ratings.shape

(59850, 9)

### Create Train/Test Split

In [41]:
# does NOT immediately compute results.
# It creates a GroupBy object (a special pandas object).

obj = final_ratings.groupby("user_id")
print("type : ", type(obj))
print("len : ", len(obj)) # number of groups / Number of unique users in final_ratings

type :  <class 'pandas.core.groupby.generic.DataFrameGroupBy'>
len :  888


In [42]:
from sklearn.model_selection import train_test_split

train_list = []
test_list = []

# groupby("user_id") splits the ratings dataframe into mini dataframes per user
# group_dataframe = ratings of ONE user
# user : user_id value
# group_dataframe : all ratings of that user (DataFrame)
for user, group_dataframe in final_ratings.groupby("user_id"):
    
    # Users with very few ratings are skipped
    if len(group_dataframe) < 5:
        continue

    # we have only passed one data set that will devide usually into 2, if we passes two as (X, y, ...) then it x --> 2, y --> that is why we see 4 
    # variables as X_train, X_test, y_train, y_test
    # 80% data frames for tratining
    train, test = train_test_split(group_dataframe, test_size=0.2, random_state=42) 
    train_list.append(train)
    test_list.append(test)

# pd.concat() = stack/merge each users group dataframes vertically (row-wise).
train_ratings = pd.concat(train_list)
test_ratings  = pd.concat(test_list)

print("Train:", train_ratings.shape)
print("Test:", test_ratings.shape)

Train: (47471, 9)
Test: (12300, 9)


### Pivot table
- this will contain user_ids with book title with ratings

In [57]:
# Build Pivot ONLY from Train Data

book_pivot = train_ratings.pivot_table(
    index="title",
    columns="user_id",
    values="ratings"
).fillna(0)
print(book_pivot.shape)
book_pivot.head(3)

(742, 858)


user_id,254,2276,2766,2977,3363,4017,6242,6251,6323,6543,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Convert pivot table to sparse matrix by removing zeros for efficient memory usage and faster computation.

In [59]:
from scipy.sparse import csr_matrix

book_sparse = csr_matrix(book_pivot)
print(book_sparse.shape)
print(book_sparse.nnz) # So out of 636,636 possible cells, only : 11,819 ratings actually exist
book_sparse[0] # Row 0 = first book (“1984”), Only 23 users rated this book, Even though there are 858 users total.

(742, 858)
11819


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23 stored elements and shape (1, 858)>

## **Model training - KNN**

In [81]:
from sklearn.neighbors import NearestNeighbors

# model has never seen test ratings

# There are different ways to find nearest neighbors, brute = Brute Force search
# Compare with every single book to find the closest ones, It calculates distance like: >>> distance(bookA, bookB) >>> for ALL books

model = NearestNeighbors(algorithm='brute')

In [82]:
# In collaborative filtering or content‑based recommendation, the input matrix (often called a user‑item matrix or item‑feature matrix) 
# tends to be very sparse. For example: Each user rates only a small fraction of all books Or each book has only a few features out of many possible ones

# Storing such a matrix as a dense array (like a NumPy array) would waste memory and slow down computations because most entries are zeros.
# The csr_matrix format stores only the non‑zero values along with their row and column indices, drastically reducing memory usage and enabling 
# efficient mathematical operations.

# NearestNeighbors from scikit‑learn can accept sparse matrices directly

model.fit(book_sparse)

NearestNeighbors(algorithm='brute')

#### Recommendation function

In [62]:
book_pivot.index # pandas Index object containing all book names

Index(['1984', '1st to Die: A Novel', '2nd Chance', '4 Blondes',
       '84 Charing Cross Road', 'A Bend in the Road', 'A Case of Need',
       'A Child Called \It\": One Child's Courage to Survive"',
       'A Civil Action', 'A Cry In The Night',
       ...
       'Winter Solstice', 'Wish You Well', 'Without Remorse',
       'Wizard and Glass (The Dark Tower, Book 4)', 'Wuthering Heights',
       'Year of Wonders', 'You Belong To Me',
       'Zen and the Art of Motorcycle Maintenance: An Inquiry into Values',
       'Zoya', '\O\" Is for Outlaw"'],
      dtype='object', name='title', length=742)

In [78]:
# get_loc() = get location
# It converts a label → numeric position
# “At what row number is this book located?”
# "Harry Potter"  →  row number 123

book_pivot.index.get_loc("Wish You Well")

733

In [93]:
# iloc = select by row number
# So this grabs the ratings vector of that book
# .values >>> Convert to numpy array

print(book_pivot.shape)
book_pivot.iloc[563, : ] # get 563 record
book_pivot.iloc[563, : ].values # KNN (scikit-learn) works with numpy arrays, not pandas

print(book_pivot.iloc[563, : ].shape)
m = book_pivot.iloc[563, : ].values.reshape(1, -1) # Reshape to 2D, ROW vector
print(m.shape)

(742, 858)
(858,)
(1, 858)


In [99]:
def recommend_books(book_name, n=6):

    # Skip unknown books
    if book_name not in book_pivot.index:
        return []

    book_idx = book_pivot.index.get_loc(book_name) # At what row number is this book located?

    distance, indices = model.kneighbors(
        # get reshaped row vector of a particular book by id that converted into numpy array from pandas series
        # since a book data in a row form not in column, So each book must be a row >> (1, -1)
        book_pivot.iloc[book_idx, : ].values.reshape(1, -1), 
        n_neighbors=n+1 # Because the closest book is the book itself thati is why n+1
    )
        
    rec_books = []
    for i in range(1, len(indices.flatten())):
        rec_books.append(book_pivot.index[indices.flatten()[i]])
        
    return rec_books

In [100]:
# This model work as follows
# if some one open a book A model look for books that in same cluster with min distance. The cluster is created using ratings of each user.
# If users read/rated book A, Book B and also Book C, then another user open/click on Book A, then we can say/ suggest some 
# other books that in same cluster with Book A with min distance to Book A, cluster creation done by actualy according to user interactions 
# here only ratings in content based system we use genre, title, category, ... but in collaborative approach we use user interaction to cluster 
# simillar vibe books

book_name = "A Bend in the Road"

recommend_books(book_name)

['Last Man Standing',
 'No Safe Place',
 'Exclusive',
 'The Cradle Will Fall',
 'Jacob Have I Loved',
 'Long After Midnight']

## **Model Evaluation**
- When we recommend books to users, how often are they actually books the user likes?
#### **evaluation must be user-based :**
- We simulate real world:

1. Hide some ratings from users
2. Recommend books to that user
3. Check if recommended books were actually liked

- This is called Train/Test split for recommender systems.


  
#### **Precision@K Intuition**
- **how accurate the recommendations are**

- Imagine we recommend K = 5 books to a user.
- Recommended → [A, B, C, D, E]
- Actual liked books → [A, C, F]
- Relevant recommendations = A, C = 2
- Precision@5 = relevant / recommended
 = 2 / 5 = 0.40
-  “40% of recommendations were good”



#### **Recall@K Intuition**

- **how much of the user’s liked books we found**

- User liked 3 books → [A, C, F]
- We found only [A, C]

- Recall@5 = found relevant / total relevant
= 2 / 3 = 0.67

-  “We found 67% of books user likes”

In [103]:
# k = how many books we recommend
# threshold = rating considered “liked”

def precision_recall_at_k(k=5, threshold=8):

    # We will compute precision/recall for each user, then average them
    precisions = []
    recalls = []

    # Evaluate recommendations user by user.
    for user in test_ratings["user_id"].unique():

        # This dataframe contains : books this user rated in the TEST set
        user_test = test_ratings[test_ratings["user_id"] == user]
        # books the user actually liked
        liked_books = user_test[user_test["ratings"] >= threshold]["title"].tolist()

        # Skip users who didn’t like anything
        if len(liked_books) == 0:
            continue
        
        # pick one book the user liked (seed book)
        seed_book = liked_books[0]

        # Generate recommendations
        recs = recommend_books(seed_book, n=k)

        # Skip if no recs
        if len(recs) == 0:
            continue

        # Find relevant recommendations
        # We compute intersection : recommended books ∩ liked books
        # recs        = [A, B, C, D, E]
        # liked_books = [B, D, F]
        # intersection = [B, D]
        # relevant = 2
        relevant = len(set(recs) & set(liked_books))

        # Compute Precision@K
        precision = relevant / k # “Out of recommended books, how many were good?”
        
        recall = relevant / len(liked_books) # “Out of all books the user likes, how many did we find?”

        # Store user results, We repeat for all users
        precisions.append(precision)
        recalls.append(recall)

    # Return overall performance, We average across users → final score
    return np.mean(precisions), np.mean(recalls)

In [104]:
precision, recall = precision_recall_at_k(k=10, threshold=7)

print("Precision@5:", precision)
print("Recall@5:", recall)

Precision@5: 0.004782608695652174
Recall@5: 0.009902901044205393
